# 02 — Image inventory

For each dry-season-year, group candidate scenes by sensor-date with AOI cloud % and coverage %, then select the single clearest full-coverage image (or flag a gap year).

In [ ]:
import os
REPO_DIR = '/content/Shoreline-Dynamics'
if os.path.isdir(REPO_DIR):
    # Already cloned this session: pull latest main so code fixes are picked
    # up (a plain clone-if-missing would keep stale code on disk).
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone https://github.com/SKPrince1911/Shoreline-Dynamics.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q earthengine-api geemap

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='shoreline-analysis')
print('GEE initialized:', ee.String('ok').getInfo())

In [ ]:
import sys; sys.path.append('.')
import importlib
import pandas as pd
import src.config, src.data
# Reload in case an older copy was imported earlier this session (config
# first, since data imports it).
importlib.reload(src.config)
importlib.reload(src.data)
from src import config, data

aoi = ee.Geometry.Polygon(config.aoi_coordinates())

# Retrieval lives in src/data.py so fixes propagate via `git pull` (the
# notebook cells you run are a separate copy from the repo files). Both
# helpers page results in small key-chunks to avoid Earth Engine's
# 'Too many concurrent aggregations' (HTTP 429) on high-sensor years.
def candidates_df(year):
    """Per sensor-date candidates for a dry-season-year, sorted by AOI cloud %."""
    return data.candidates_dataframe(year, aoi)

def best_df(year):
    """The single selected image (or gap record) for a dry-season-year."""
    feat = data.select_best(year, aoi)  # ee.Feature of plain scalars
    return pd.DataFrame([feat.getInfo().get('properties', {})])

## Test year 1995

In [ ]:
candidates_df(1995)

In [ ]:
best_df(1995)

## Test year 2010

In [ ]:
candidates_df(2010)

In [ ]:
best_df(2010)

## Test year 2022 (high-sensor year: L7 + L8 + L9 + S2)

In [ ]:
candidates_df(2022)

In [ ]:
best_df(2022)